# Week 7 companion - Modal forecasting of a real cylinder wake
<!-- MIE690A article-aligned validation v4 -->
**Development companion:** local Run All is tested; main-branch Colab becomes
available only after merge. No new release or DOI is implied.

Compare projected DMD with a freshly trained nonlinear modal baseline, persistence,
and POD representation floors. Prerequisites: POD, complex eigenvalues and Week 7
LBM; allow 60-75 minutes. Lecture: `lectures/week07_modal_forecasting.pdf`.

All fields come from earlier author-generated CFD, not synthetic wake formulas.
The 12-node-per-diameter LBM is educational, not grid independent. This wake has
vortical structures but no shock. DMD forecasts a field; it does not identify cores.
<!-- FLOWMLLAB_COLAB_LAUNCH_V1 -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ehsan-Roohi/FlowMLLab/blob/main/notebooks/week07/W7_Lab2_Modal_Forecasting.ipynb)


In [ ]:
# FLOWMLLAB_COLAB_BOOTSTRAP_V1
# In Colab this cell obtains the complete repository and installs the tested package.
# In a local checkout it leaves the active environment and working directory unchanged.
from pathlib import Path as _FlowMLLabPath
import os as _flowmllab_os
import subprocess as _flowmllab_subprocess
import sys as _flowmllab_sys

if "google.colab" in _flowmllab_sys.modules or _flowmllab_os.environ.get("COLAB_RELEASE_TAG"):
    _flowmllab_root = _FlowMLLabPath("/content/FlowMLLab")
    if not (_flowmllab_root / ".git").is_dir():
        _flowmllab_subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "https://github.com/Ehsan-Roohi/FlowMLLab.git", str(_flowmllab_root),
            ],
            check=True,
        )
    _flowmllab_subprocess.run(
        [
            _flowmllab_sys.executable, "-m", "pip", "install", "-q", "-e",
            f"{_flowmllab_root}[test]",
        ],
        check=True,
    )
    _flowmllab_notebook_dir = _flowmllab_root / "notebooks/week07"
    _flowmllab_os.chdir(_flowmllab_notebook_dir)
    for _flowmllab_path in (_flowmllab_root, _flowmllab_notebook_dir):
        if str(_flowmllab_path) not in _flowmllab_sys.path:
            _flowmllab_sys.path.insert(0, str(_flowmllab_path))
    print("FlowMLLab ready:", _flowmllab_root)

from pathlib import Path
import sys, hashlib, tempfile
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'flowmllab/modal_experiments.py').is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'qa'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from threadpoolctl import threadpool_limits
from flowmllab.modal_tools import *
from flowmllab.field_metrics import field_metrics, temporal_spectrum
from flowmllab.modal_experiments import PLAN, load_cases, forecast_experiment, sensor_experiment
from run_modal_labs import figures
tracked_evidence = [p for folder in ('data/modal_labs','results/modal_labs') for p in (ROOT/folder).rglob('*') if p.is_file()]
before = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in tracked_evidence}
cases = load_cases(ROOT/'data/modal_labs')
display(PLAN)
print('Checksummed author LBM; previously inspected cases, not a new blind CFD study.')

## 1. Derive and test the row-vector convention
For training POD rows $a_k$, let $X=[a_0;\ldots;a_{m-2}]$ and
$Y=[a_1;\ldots;a_{m-1}]$. Solve $A=X^\dagger Y$, so $a_{k+1}=a_kA$.
Reconstruct with the frozen training mean and basis. For eigenvalue $\lambda$,
$f=\arg(\lambda)/(2\pi\Delta t)$ and $\sigma=\log|\lambda|/\Delta t$.
Frequency is subject to sampling and the principal logarithm branch.

Verify a known oscillator before interpreting wake eigenvalues. Its angular
frequency is 2, so cycles per unit time must be $1/\pi$.

In [ ]:
dt = .02
t = np.arange(400)*dt
oscillator = np.column_stack((np.cos(2*t), np.sin(2*t)))
operator = fit_dmd(oscillator[:200])
assert np.allclose(rollout_dmd(operator, oscillator[199], 200), oscillator[200:], atol=1e-12)
assert np.isclose(max(m['frequency'] for m in dmd_modes(operator,dt)), 1/np.pi)
display(pd.DataFrame(dmd_modes(operator,dt)))

## 2. Forecast, do not assimilate the answer
Re110 vorticity: training frames [0,160), validation [160,210), test [210,281).
Fit centered POD only on training. Choose DMD rank among 2/4/6/8 by validation
field error. Initialize at frame159 and advance through both held windows
without resetting. This tests future time in one case, not a new Reynolds number.

The fresh MLP uses eight POD coordinates, four-frame history, two 32-unit tanh
layers and fixed seed17. Scaling and supervised pairs use training only.
It has more initial history than one-state DMD; report that distinction.
Persistence repeats the last training field; the mean repeats the training mean.
Full-field POD projections see target fields and are oracle floors, not forecasts.
Do not compare to archived phase-decoder results with different information/splits.

In [ ]:
# Fresh CPU fits, not cached retained predictions. Limit BLAS threads for reproducibility.
with threadpool_limits(limits=1):
    forecast, predictions = forecast_experiment(cases)
    sensing, examples = sensor_experiment(cases)
scratch = Path(tempfile.mkdtemp(prefix='flowmllab-modal-'))
figures(scratch, cases, forecast, predictions, sensing, examples)
print('Exploratory figure output:', scratch)

In [ ]:
print('Validation-selected DMD:', forecast['selected_dmd'])
display(pd.DataFrame({k:{'validation_l2':v['validation']['relative_l2'],
    'test_l2':v['test']['relative_l2'], 'worst_test_frame':v['test']['worst_frame_relative_l2']}
    for k,v in forecast['methods'].items()}).T)
display(forecast['methods']['MLP-r8'])
display(Image(filename=str(scratch/'forecast_fields.png')))
display(Image(filename=str(scratch/'forecast_audit.png')))

## 3. A frequency estimate is not a resolution guarantee
The source full-history force Strouhal is a diagnostic only; it was never used
to train or select a model. Inspect all selected-DMD eigenvalues and distinguish
wake modes, harmonics and possible box/acoustic modes. Do not automatically
declare the largest frequency to be shedding.

At a geometrically selected probe near (x/D,y/D)=(4,0.5), compare mean-removed,
Hann-windowed one-sided temporal spectra. Report the actual FFT spacing and
phase error at the reference peak. The test window is only about 7.40 D/U,
giving spacing about 0.135 U/D: too short for a precise Strouhal measurement.
A model-based eigenfrequency does not improve the raw sampling resolution.

The Week 5 SINDy result is also retained, including failed candidates. Its
rank-two field limitation is not evidence that all sparse dynamics fail.

In [ ]:
selected = forecast['selected_dmd']
display(pd.DataFrame(forecast['methods'][selected]['modes']))
print('Full-history force Strouhal (diagnostic only):', forecast['source_full_history_strouhal_diagnostic_only'])
print('Test duration:', forecast['test_duration'])
display(pd.DataFrame(forecast['spectra']).T)
display(pd.DataFrame(forecast['sindy_candidates']).T)

## A shared metric contract (PDEBench-inspired, not identical scores)
Report $\|q-\hat q\|_{2,w}/\|q\|_{2,w}$, area-weighted RMSE, maximum error,
worst-frame relative error, ROI-edge error, and error of the supplied scalar integral.
An ROI edge is **not a physical wall**. An integral of vorticity or velocity is
**not automatically mass conservation**. Declare weights, units and geometry.
For zero reference norm, relative error is undefined (`None`), not epsilon-regularized.

Test a known offset before trusting the CFD score: adding one to a field of two
must give relative L2 = 0.5 and RMSE = 1. With 20 cells of area 0.25, the
absolute scalar-integral error must be 5. These are numerical identities, not fitted tolerances.

In [ ]:
truth = np.full((3,4,5), 2.0)
metrics = field_metrics(truth+1, truth, np.full((4,5), .25))
assert np.isclose(metrics['relative_l2'], .5)
assert np.isclose(metrics['rmse'], 1)
assert np.isclose(metrics['mean_absolute_scalar_integral_error'], 5)
assert field_metrics(np.ones_like(truth), 0*truth)['relative_l2'] is None
display(metrics)

## 4. Interpret and extend without changing the retained test
Explain the gap between the eight-mode POD oracle and DMD. Distinguish
discretization error, representation error and autonomous dynamics error.
Why would resetting to a true state at frame210 make the forecast easier?
Propose a new-Re case and a longer-duration test, with thresholds frozen before
seeing either. For a noise-robust DMD study, add a declared observation-noise
protocol and a matched robust method; this exercise does not implement PyDMD's
noise-aware variants. Do not infer vortex-segmentation performance from a good
forecast contour. Common contour levels are for display only; no data smoothing
or resolution enhancement is used.

## Sources, provenance and submission
The data are earlier **author-generated FlowMLLab LBM cases**, published in
[cylinder-cfd-v1](https://github.com/Ehsan-Roohi/FlowMLLab/releases/tag/cylinder-cfd-v1).
They are not copied package examples or data from the separate hypersonic DSMC article.
See `data/modal_labs/README.md` and its original/derived file hashes.

Original textbook implementations were inspired by
[PyDMD](https://github.com/PyDMD/PyDMD),
[PySensors](https://github.com/dynamicslab/pysensors),
[PySINDy](https://github.com/dynamicslab/pysindy), and
[PDEBench](https://github.com/pdebench/PDEBench).
No code or figures were copied; these are not wrappers, complete replacements,
or a claim of exact equivalence to the packages' advanced algorithms.

Submit the complete split, selected settings, all candidate/seed scores, one
failure explanation, and one proposed **new** validation experiment. Do not
retune on the retained test or overwrite reference evidence. Short CPU runtime
reflects reuse of already-generated CFD, not a fresh high-fidelity simulation.

In [ ]:
after = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in tracked_evidence}
assert after == before, 'Notebook modified retained data/evidence'
print('PASS: retained data and evidence unchanged.')